In [14]:
import numpy as np
import time
import pandas as pd
import os
import rasterio as rio
import cripser
from gtda.diagrams import PersistenceLandscape, PersistenceImage, BettiCurve
from scipy.interpolate import interp1d
from pathlib import Path
import tempfile
import shutil

# Euchar library for ECC computation
import euchar.utils
import euchar.curve

In [15]:
NUM_TILES = 100

PROJECT_ROOT = Path("../..").resolve()
data_folder_path = PROJECT_ROOT / 'data' / 'earthscape' / 'earthscape_warren_rockfield'
labels_path = data_folder_path / 'labels.csv'
patches_folder_path = data_folder_path / 'patches'

# Output directory configuration
output_dir = PROJECT_ROOT / 'data' / 'processed' / 'minimal_exploration'
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

label_map = {
    'af1': 'artificial_fill',
    'Qal': 'alluvium',
    'Qaf': 'alluvial_fans',
    'Qat': 'alluvial_terraces',
    'Qc': 'colluvium',
    'Qca': 'colluvial_accumulations',
    'Qr': 'residuum'
}

df = pd.read_csv(labels_path).iloc[:NUM_TILES].rename(columns=label_map)

df

Output directory: D:\dev\karst-terrain-classification\data\processed\minimal_exploration


,patch_id,artificial_fill,alluvium,alluvial_fans,alluvial_terraces,colluvium,colluvial_accumulations,residuum
0,256_50_1,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,256_50_2,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,256_50_3,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,256_50_4,1.0,1.0,0.0,0.0,0.0,0.0,1.0
4,256_50_5,1.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...
95,256_50_130,1.0,1.0,0.0,0.0,0.0,1.0,1.0
96,256_50_131,1.0,1.0,0.0,0.0,0.0,1.0,1.0
97,256_50_132,0.0,1.0,0.0,0.0,0.0,1.0,1.0
98,256_50_133,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [16]:
patch_ids = df['patch_id']
x_dem_filenames = [patches_folder_path / f"{pid}_dem.tif" for pid in patch_ids]

patch_ids

0       256_50_1
1       256_50_2
2       256_50_3
3       256_50_4
4       256_50_5
         ...    
95    256_50_130
96    256_50_131
97    256_50_132
98    256_50_133
99    256_50_134
Name: patch_id, Length: 100, dtype: object

In [17]:
readable_label_cols = list(label_map.values())
labels_df = df[readable_label_cols]
y_labels = labels_df.to_numpy()

labels_df

,artificial_fill,alluvium,alluvial_fans,alluvial_terraces,colluvium,colluvial_accumulations,residuum
0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,1.0,1.0,0.0,0.0,0.0,0.0,1.0
4,1.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...
95,1.0,1.0,0.0,0.0,0.0,1.0,1.0
96,1.0,1.0,0.0,0.0,0.0,1.0,1.0
97,0.0,1.0,0.0,0.0,0.0,1.0,1.0
98,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [18]:
print("--- Verification (First 3 Entries) ---")
for i in range(3):
    print(f"Entry {i}:")
    print(f"\tX file: {x_dem_filenames[i]}")
    print(f"\ty fields: {list(labels_df.columns)}")
    print(f"\ty label: {y_labels[i]}")

--- Verification (First 3 Entries) ---
Entry 0:
	X file: D:\dev\karst-terrain-classification\data\earthscape\earthscape_warren_rockfield\patches\256_50_1_dem.tif
	y fields: ['artificial_fill', 'alluvium', 'alluvial_fans', 'alluvial_terraces', 'colluvium', 'colluvial_accumulations', 'residuum']
	y label: [1. 1. 0. 0. 0. 0. 1.]
Entry 1:
	X file: D:\dev\karst-terrain-classification\data\earthscape\earthscape_warren_rockfield\patches\256_50_2_dem.tif
	y fields: ['artificial_fill', 'alluvium', 'alluvial_fans', 'alluvial_terraces', 'colluvium', 'colluvial_accumulations', 'residuum']
	y label: [1. 1. 0. 0. 0. 0. 1.]
Entry 2:
	X file: D:\dev\karst-terrain-classification\data\earthscape\earthscape_warren_rockfield\patches\256_50_3_dem.tif
	y fields: ['artificial_fill', 'alluvium', 'alluvial_fans', 'alluvial_terraces', 'colluvium', 'colluvial_accumulations', 'residuum']
	y label: [1. 1. 0. 0. 0. 0. 1.]


In [19]:
def atomic_save(filepath, data, **save_kwargs):
    """
    Atomically save numpy array to prevent corruption on crashes.
    
    Writes to a temporary file first, then atomically renames to target path.
    This ensures the target file is never in a partially-written state.
    
    Args:
        filepath: Path or str, target file path
        data: Data to save (numpy array or dict for pickle)
        **save_kwargs: Additional arguments to pass to np.save()
    """
    filepath = Path(filepath)
    
    # Create temporary file in the same directory (ensures same filesystem for atomic rename)
    temp_fd, temp_path = tempfile.mkstemp(
        dir=filepath.parent,
        prefix=f'.tmp_{filepath.stem}_',
        suffix=filepath.suffix
    )
    
    try:
        # Close the file descriptor (np.save will open it)
        os.close(temp_fd)
        
        # Save to temporary file
        np.save(temp_path, data, **save_kwargs)
        
        # Atomic rename (POSIX guarantees atomicity)
        # On Windows, need to remove target first if it exists
        if os.name == 'nt' and filepath.exists():
            filepath.unlink()
        
        shutil.move(temp_path, filepath)
        
    except Exception:
        # Clean up temp file on error
        if os.path.exists(temp_path):
            os.unlink(temp_path)
        raise


def _sanitize_value(val, default=0.0):
    """Replace inf/NaN with default value."""
    return val if np.isfinite(val) else default


def _clean_tile(tile):
    """
    Clean DEM tile by replacing inf/NaN values.
    
    Args:
        tile: 2D numpy array (DEM elevation data)
    
    Returns:
        Cleaned tile or None if entire tile is invalid
    """
    tile_clean = tile.copy().astype(np.float64)
    
    if np.isfinite(tile_clean).all():
        return tile_clean
    
    finite_mask = np.isfinite(tile_clean)
    if not finite_mask.any():
        return None
    
    # Replace invalid values with minimum (topologically sound approach)
    tile_min = tile_clean[finite_mask].min()
    tile_clean[~finite_mask] = tile_min
    
    return tile_clean


def _prepare_persistence_diagrams(pd):
    """
    Extract and prepare persistence diagrams for H0 and H1 from cripser output.
    
    Args:
        pd: Persistence diagram from cripser.computePH (sanitized)
    
    Returns:
        Tuple of (diagram_0, diagram_1) as numpy arrays in giotto-tda format,
        or None if empty
    """
    if pd is None or len(pd) == 0:
        return None
    
    # Extract persistence intervals by dimension
    pairs_0 = pd[pd[:, 0] == 0][:, 1:3]  # H0: connected components
    pairs_1 = pd[pd[:, 0] == 1][:, 1:3]  # H1: loops/holes
    
    # Remove infinite persistence (cripser uses float_max, not actual inf)
    # Filter out both inf/nan AND extreme values close to float_max
    EXTREME_THRESHOLD = 1e100  # Anything above this is considered "infinite persistence"
    
    def is_valid_persistence(pairs):
        """Check if persistence pairs are finite and not extreme"""
        is_finite = np.isfinite(pairs).all(axis=1)
        is_not_extreme = (np.abs(pairs) < EXTREME_THRESHOLD).all(axis=1)
        return is_finite & is_not_extreme
    
    pairs_0_finite = pairs_0[is_valid_persistence(pairs_0)]
    pairs_1_finite = pairs_1[is_valid_persistence(pairs_1)]
    
    # Add homology dimension column (giotto-tda format)
    diagram_0 = (
        np.column_stack([pairs_0_finite, np.zeros(len(pairs_0_finite))])
        if len(pairs_0_finite) > 0
        else np.array([[0, 0, 0]])  # Dummy point for empty diagram
    )
    
    diagram_1 = (
        np.column_stack([pairs_1_finite, np.ones(len(pairs_1_finite))])
        if len(pairs_1_finite) > 0
        else np.array([[0, 0, 1]])  # Dummy point for empty diagram
    )
    
    return diagram_0, diagram_1


def sanitize_persistence_diagram(pd):
    """
    Clean persistence diagram by removing inf/NaN values.
    
    Args:
        pd: Persistence diagram from cripser.computePH
        
    Returns:
        Cleaned persistence diagram with finite values only
    """
    if pd is None or len(pd) == 0:
        return pd
    
    # Remove rows with any inf or NaN values
    finite_mask = np.isfinite(pd).all(axis=1)
    return pd[finite_mask]


def compute_traditional_features(patch_id, dir):
    """
    Compute traditional geomorphometric features from pre-computed rasters.
    
    Robustly handles missing files, inf, and NaN values.
    
    Args:
        patch_id: Patch identifier string
        dir: Directory containing feature rasters
    
    Returns:
        Feature vector with statistics (mean, std, min, max, median) for each raster
    """
    FEATURES_KERNELS = ['ep', 'stdslope']
    KERNELS = ['5', '11', '21', '51', '101', '201']
    FEATURES_RESOLUTIONS = ['plancurv', 'procurv', 'slope']
    RESOLUTIONS = ['5', '10', '20', '50', '100', '200']
    
    all_suffixes = (
        [f"{f}_{k}" for f in FEATURES_KERNELS for k in KERNELS] +
        [f"{f}_{r}" for f in FEATURES_RESOLUTIONS for r in RESOLUTIONS]
    )
    
    all_stats = []
    
    for suffix in all_suffixes:
        feature_path = dir / f"{patch_id}_{suffix}.tif"
        try:
            with rio.open(feature_path) as src:
                raster = src.read(1).astype(np.float64)
                valid_mask = np.isfinite(raster)
                
                if valid_mask.any():
                    valid_data = raster[valid_mask]
                    all_stats.extend([
                        _sanitize_value(np.mean(valid_data)),
                        _sanitize_value(np.std(valid_data)),
                        _sanitize_value(np.min(valid_data)),
                        _sanitize_value(np.max(valid_data)),
                        _sanitize_value(np.median(valid_data))
                    ])
                else:
                    all_stats.extend([0.0] * 5)
        except Exception:
            # Missing file or read error - use zeros
            all_stats.extend([0.0] * 5)
    
    return np.array(all_stats, dtype=np.float64)

In [20]:
print("Pre-computing the euchar 2D optimization vector (runs once)...")
try:
    # This is the one-time calculation
    EUC_VEC_2D = euchar.utils.vector_all_euler_changes_in_2D_images()
    print("✓ Pre-computation complete. 'EUC_VEC_2D' is ready.")
except Exception as e:
    print(f"CRITICAL ERROR: Failed to pre-compute euchar vector: {e}")
    print("The 'compute_ecc_features' function will fail until this is fixed.")
    EUC_VEC_2D = None
    raise RuntimeError("Failed to pre-compute euchar vector") from e


def compute_ecc_features(tile, vector_2D_changes=None, n_samples=50, n_steps=256):
    """
    Compute Euler Characteristic Curve from DEM tile using the optimal 'euchar' library.
    
    This function implements the "optimal path" by:
    1. Using the fast C++ 'euchar' backend.
    2. Accepting a pre-computed 'vector_2D_changes' to eliminate redundant computation.
    3. Using the topologically-sound 'np.min()' strategy for NoData.
    4. Implementing robust interpolation and sanitization.
    
    Args:
        tile: 2D numpy array (DEM elevation data).
        vector_2D_changes: Pre-computed vector from euchar (uses global EUC_VEC_2D if None).
        n_samples: Number of sample points for the final ECC vector.
        n_steps: Number of integer steps for discretization (default 256).
    
    Returns:
        ECC vector of length n_samples (guaranteed finite).
    """
    # Use global EUC_VEC_2D if not provided
    if vector_2D_changes is None:
        vector_2D_changes = EUC_VEC_2D
    
    if vector_2D_changes is None:
        raise ValueError("vector_2D_changes cannot be None. Run pre-computation cell first.")
    
    # Clean input tile
    tile_clean = _clean_tile(tile)
    if tile_clean is None:
        print("  Warning: Tile is entirely non-finite, returning zero ECC vector")
        return np.zeros(n_samples, dtype=np.float64)
    
    try:
        # Discretize for euchar (requires integer inputs)
        min_val, max_val = tile_clean.min(), tile_clean.max()
        
        # Handle edge case: "flat" tile (all one value)
        if min_val == max_val:
            # The EC of a single connected component is 1
            return np.full(n_samples, 1.0, dtype=np.float64)

        discretized_image = np.interp(
            tile_clean, (min_val, max_val), (0, n_steps - 1)
        ).astype(np.int32)
        
        # Compute ECC (The Fast Way)
        ecc_values = euchar.curve.image_2D(
            image=discretized_image,
            vector_of_euler_changes_2D=vector_2D_changes,
            max_intensity=n_steps - 1
        )
        
        # Interpolate to n_samples
        original_thresholds = np.arange(n_steps)
        
        f = interp1d(
            original_thresholds,
            ecc_values,
            kind='linear',
            bounds_error=False,
            fill_value=(ecc_values[0], ecc_values[-1])
        )
        
        sample_thresholds = np.linspace(0, n_steps - 1, n_samples)
        ecc_vector = f(sample_thresholds)
        
        # Final sanitization
        ecc_vector = np.where(np.isfinite(ecc_vector), ecc_vector, 0.0)
        
        return ecc_vector.astype(np.float64)
        
    except Exception as e:
        # Log the error instead of silently returning zeros
        print(f"  ERROR in ECC computation: {type(e).__name__}: {e}")
        print(f"  Returning zero vector for this tile")
        return np.zeros(n_samples, dtype=np.float64)

Pre-computing the euchar 2D optimization vector (runs once)...
✓ Pre-computation complete. 'EUC_VEC_2D' is ready.


In [21]:
def compute_ph_features(tile=None, target_dim=1100, pd=None):
    """
    Compute persistent homology features using Cubical Ripser (3-4x faster than GUDHI).
    Uses giotto-tda's PersistenceLandscape for vectorization (C++ backend, sklearn-compatible).
    
    Robustly handles edge cases and ensures finite output.
    
    Args:
        tile: 2D numpy array (DEM elevation data) - optional if pd is provided
        target_dim: Expected output dimension (default 1100)
        pd: Pre-computed persistence diagram (from cripser.computePH) - optional
    
    Returns:
        Landscape vector of length target_dim (guaranteed finite)
    """
    try:
        # Compute persistent homology only if not provided
        if pd is None:
            if tile is None:
                raise ValueError("Either 'tile' or 'pd' must be provided")
            pd = cripser.computePH(tile.astype(np.float64), maxdim=1)
        
        # Sanitize and prepare persistence diagrams
        pd = sanitize_persistence_diagram(pd)
        diagrams = _prepare_persistence_diagrams(pd)
        
        if diagrams is None:
            return np.zeros(target_dim, dtype=np.float64)
        
        diagram_0, diagram_1 = diagrams
        
        # Create separate diagrams for H0 and H1
        diagrams_0 = diagram_0[np.newaxis, :, :]  # Shape: (1, n_points_0, 3)
        diagrams_1 = diagram_1[np.newaxis, :, :]  # Shape: (1, n_points_1, 3)
        
        # Configure persistence landscapes
        n_landscapes_0 = 5
        n_landscapes_1 = 6
        resolution = 100
        
        # Create PersistenceLandscape transformers
        landscaper_0 = PersistenceLandscape(n_layers=n_landscapes_0, n_bins=resolution)
        landscaper_1 = PersistenceLandscape(n_layers=n_landscapes_1, n_bins=resolution)
        
        # Transform to landscapes
        vec_0 = landscaper_0.fit_transform(diagrams_0)
        vec_1 = landscaper_1.fit_transform(diagrams_1)
        
        # Concatenate landscapes from both dimensions
        final_vector = np.concatenate([vec_0.flatten(), vec_1.flatten()])
        
        # Final sanitization
        final_vector = np.where(np.isfinite(final_vector), final_vector, 0.0)
        
        return final_vector.astype(np.float64)
        
    except Exception:
        return np.zeros(target_dim, dtype=np.float64)

In [22]:
def compute_betti_curves(tile=None, n_samples=100, pd=None):
    """
    Compute Betti Curves from persistent homology using giotto-tda.
    
    This computes actual Betti numbers (topological invariants) at different 
    filtration values, unlike ECC which computes Euler characteristics 
    (a less stable, less informative measure).
    
    Robustly handles edge cases and ensures finite output.
    
    Args:
        tile: 2D numpy array (DEM elevation data) - optional if pd is provided
        n_samples: Number of sample points for each Betti curve
        pd: Pre-computed persistence diagram (from cripser.computePH) - optional
    
    Returns:
        Concatenated Betti curve vectors [β0 curve, β1 curve] (guaranteed finite)
    """
    try:
        # Compute persistent homology only if not provided
        if pd is None:
            if tile is None:
                raise ValueError("Either 'tile' or 'pd' must be provided")
            pd = cripser.computePH(tile.astype(np.float64), maxdim=1)
        
        # Sanitize and prepare persistence diagrams
        pd = sanitize_persistence_diagram(pd)
        diagrams = _prepare_persistence_diagrams(pd)
        
        if diagrams is None:
            return np.zeros(n_samples * 2, dtype=np.float64)
        
        diagram_0, diagram_1 = diagrams
        
        # Combine into single diagram
        diagram = np.vstack([diagram_0, diagram_1])
        
        # Add batch dimension (giotto-tda expects batch of diagrams)
        diagrams = diagram[np.newaxis, :, :]  # Shape: (1, n_points, 3)
        
        # Create BettiCurve transformer
        betti_curve = BettiCurve(n_bins=n_samples)
        
        # Transform to Betti curves
        betti_vector = betti_curve.fit_transform(diagrams)
        
        # Final sanitization
        betti_vector = np.where(np.isfinite(betti_vector), betti_vector, 0.0)
        
        return betti_vector.flatten().astype(np.float64)
        
    except Exception:
        return np.zeros(n_samples * 2, dtype=np.float64)

In [23]:
def compute_persistence_images(tile=None, sigma=2.0, resolution=50, pd=None):
    """
    Compute Persistence Images from persistent homology using giotto-tda.
    
    Persistence Images convert the persistence diagram into a 2D "image" by:
    1. Transforming (birth, death) -> (birth, persistence)
    2. Applying a weighted 2D Gaussian to each point
    3. Discretizing onto a pixel grid
    
    This representation is ideal for CNNs but can also be flattened for
    traditional classifiers (RF, SVM, etc.) for direct comparison with Landscapes.
    
    Robustly handles edge cases and ensures finite output.
    
    Args:
        tile: 2D numpy array (DEM elevation data) - optional if pd is provided
        sigma: Bandwidth of Gaussian kernel (default 2.0)
        resolution: Grid resolution (resolution x resolution image per homology dimension)
        pd: Pre-computed persistence diagram (from cripser.computePH) - optional
    
    Returns:
        Concatenated flattened image vectors [img_0_flat, img_1_flat] (guaranteed finite)
        For resolution=50, returns a 5000-dimensional vector (2 x 50x50)
    """ 
    try:
        # Compute persistent homology only if not provided
        if pd is None:
            if tile is None:
                raise ValueError("Either 'tile' or 'pd' must be provided")
            pd = cripser.computePH(tile.astype(np.float64), maxdim=1)
        
        # Sanitize and prepare persistence diagrams
        pd = sanitize_persistence_diagram(pd)
        diagrams = _prepare_persistence_diagrams(pd)
        
        if diagrams is None:
            return np.zeros(resolution * resolution * 2, dtype=np.float64)
        
        diagram_0, diagram_1 = diagrams
        
        # Combine into single diagram
        diagram = np.vstack([diagram_0, diagram_1])
        
        # Add batch dimension (giotto-tda expects batch of diagrams)
        diagrams = diagram[np.newaxis, :, :]  # Shape: (1, n_points, 3)
        
        # Create PersistenceImage transformer
        pimgr = PersistenceImage(n_bins=resolution, sigma=sigma)
        
        # Transform to images
        images = pimgr.fit_transform(diagrams)
        
        # Final sanitization
        images = np.where(np.isfinite(images), images, 0.0)
        
        return images.flatten().astype(np.float64)
        
    except Exception:
        return np.zeros(resolution * resolution * 2, dtype=np.float64)

In [ ]:
def compute_ph_directional(tile, n_samples=50, n_directions=8, gradient_strength=0.1):
    """
    Compute directional persistent homology features using the "Flashlight Method".
    
    This is a robust version that correctly sanitizes persistence diagrams
    containing 'inf' values, which are expected from this filtration method.
    This prevents `RuntimeWarning` and `NaN` values in the output vectors.
    
    Args:
        tile: 2D numpy array (DEM elevation data).
        n_samples: Number of sample points for each Betti curve (default 50).
        n_directions: Number of directions to sample (default 8).
        gradient_strength: (Unused) Kept for API compatibility.
    
    Returns:
        Concatenated Betti curve vectors from all directions (guaranteed finite).
    """
    try:
        # 1. Define the correct filtration range for the gradient
        GRADIENT_MIN = -1.0
        GRADIENT_MAX = 1.0
        # Define a finite replacement for "infinite" persistence
        INF_REPLACEMENT = GRADIENT_MAX + 0.01 
        
        # 2. Clean input tile
        tile_clean = _clean_tile(tile)
        if tile_clean is None:
            return np.zeros(n_samples * 2 * n_directions, dtype=np.float64)
        
        # 3. Create Binary Image (the "object")
        tile_mean = tile_clean[np.isfinite(tile_clean)].mean()
        binary_shape = (tile_clean > tile_mean)

        # 4. Create coordinate grids (for the "flashlight" gradients)
        rows, cols = tile_clean.shape
        y_coords = np.linspace(-1, 1, rows)[:, np.newaxis]
        x_coords = np.linspace(-1, 1, cols)[np.newaxis, :]
        
        angles = np.linspace(0, 2 * np.pi, n_directions, endpoint=False)
        all_directional_features = []
        
        betti_curve_transformer = BettiCurve(n_bins=n_samples)

        for angle in angles:
            # 5. Create the directional gradient (the "height function")
            directional_gradient = (
                x_coords * np.cos(angle) + y_coords * np.sin(angle)
            )
            
            # 6. Create the filtration image (Flashlight Method)
            filtration_image = np.where(
                binary_shape, 
                directional_gradient, 
                np.inf
            )

            # 7. Compute PH
            pd_raw = cripser.computePH(filtration_image.astype(np.float64), maxdim=1)
            
            # 8. Robustly Sanitize the Diagram
            if pd_raw is None or len(pd_raw) == 0:
                all_directional_features.append(np.zeros(n_samples * 2, dtype=np.float64))
                continue
            
            # Filter for valid features: finite birth, dim 0 or 1
            valid_mask = (
                (np.isfinite(pd_raw[:, 1])) &
                ((pd_raw[:, 0] == 0) | (pd_raw[:, 0] == 1))
            )
            pd_clean = pd_raw[valid_mask].copy()

            # Replace infinite deaths with our finite replacement value
            inf_death_mask = np.isinf(pd_clean[:, 2])
            pd_clean[inf_death_mask, 2] = INF_REPLACEMENT

            # Filter out any remaining NaNs in death (should be rare)
            finite_death_mask = np.isfinite(pd_clean[:, 2])
            pd_clean = pd_clean[finite_death_mask]

            # Filter for positive persistence (death > birth)
            persistence_mask = pd_clean[:, 2] > pd_clean[:, 1]
            pd_final = pd_clean[persistence_mask]

            # 9. Prepare for giotto-tda (replaces _prepare_persistence_diagrams)
            pairs_0 = pd_final[pd_final[:, 0] == 0][:, 1:3]
            pairs_1 = pd_final[pd_final[:, 0] == 1][:, 1:3]
            
            diagram_0 = np.column_stack([pairs_0, np.zeros(len(pairs_0))])
            diagram_1 = np.column_stack([pairs_1, np.ones(len(pairs_1))])
            
            # Combine into single diagram
            diagram = np.vstack([diagram_0, diagram_1])
            
            if diagram.size == 0:
                 all_directional_features.append(np.zeros(n_samples * 2, dtype=np.float64))
                 continue

            diagram_batch = diagram[np.newaxis, :, :] # Add batch dimension

            # 10. Vectorize
            betti_vector = betti_curve_transformer.fit_transform(diagram_batch).flatten()
            betti_vector = np.where(np.isfinite(betti_vector), betti_vector, 0.0)
            all_directional_features.append(betti_vector)
        
        # 11. Concatenate all directions
        final_vector = np.concatenate(all_directional_features).astype(np.float64)
        
        expected_dim = n_samples * 2 * n_directions
        if final_vector.shape[0] != expected_dim:
             padded_vector = np.zeros(expected_dim, dtype=np.float64)
             pad_len = min(expected_dim, final_vector.shape[0])
             padded_vector[:pad_len] = final_vector[:pad_len]
             return padded_vector
             
        return final_vector

    except Exception as e:
        print(f"ERROR in compute_ph_directional (Flashlight): {e}. Returning zero vector.")
        return np.zeros(n_samples * 2 * n_directions, dtype=np.float64)

In [25]:
from collections import defaultdict

print(f"Computing features for {len(x_dem_filenames)} tiles...")

# Storage for all feature types
features = defaultdict(list)
timing = defaultdict(list)

# Feature extraction pipeline
for i, (patch_id, dem_filename) in enumerate(zip(patch_ids, x_dem_filenames)):
    print(f"\n{'='*60}")
    print(f"Tile {i+1}/{len(x_dem_filenames)} ({patch_id})")
    print(f"{'='*60}")
    
    # Load DEM tile once for all topological methods
    with rio.open(dem_filename) as src:
        dem_tile = src.read(1)
    
    # === BASELINE: Traditional Geomorphometric Features ===
    start = time.time()
    feat = compute_traditional_features(patch_id, patches_folder_path)
    timing['traditional'].append(time.time() - start)
    features['traditional'].append(feat)
    print(f"✓ Traditional ({len(feat)}d): {timing['traditional'][-1]:.3f}s")
    
    # === Euler Characteristic Curves ===
    for n_samples, key in [(50, 'ecc50'), (200, 'ecc200')]:
        start = time.time()
        feat = compute_ecc_features(dem_tile, n_samples=n_samples)
        timing[key].append(time.time() - start)
        features[key].append(feat)
        print(f"✓ ECC-{n_samples}d: {timing[key][-1]:.3f}s")
    
    # === Optimized TDA Pipeline: Compute PH once, reuse for all vectorizations ===
    print("\nTDA Pipeline (Optimized):")
    
    # Step 1: Compute persistent homology ONCE
    start = time.time()
    pd = cripser.computePH(dem_tile.astype(np.float64), maxdim=1)
    timing['ph_compute'].append(time.time() - start)
    print(f"  → PH Computation: {timing['ph_compute'][-1]:.3f}s")
    
    # Step 2: Multiple vectorizations using the same PH
    vectorizations = [
        (100, 'betti100', compute_betti_curves, "Betti-200d"),
        (200, 'betti200', compute_betti_curves, "Betti-400d"),
        (None, 'landscape', compute_ph_features, "Landscapes-1100d"),
        ((2.0, 50), 'persimage', compute_persistence_images, "PersImages-5000d"),
    ]
    
    for params, key, func, desc in vectorizations:
        start = time.time()
        
        # Call function with appropriate parameters
        if key == 'landscape':
            feat = func(pd=pd)
        elif key == 'persimage':
            sigma, resolution = params
            feat = func(pd=pd, sigma=sigma, resolution=resolution)
        else:
            feat = func(pd=pd, n_samples=params)
        
        timing[key].append(time.time() - start)
        features[key].append(feat)
        print(f"  → {desc} vectorization: {timing[key][-1]:.3f}s")
    
    # Summary: Total TDA time
    tda_keys = ['ph_compute', 'betti100', 'betti200', 'landscape', 'persimage']
    total_tda_time = sum(timing[k][-1] for k in tda_keys)
    print(f"  → Total TDA: {total_tda_time:.3f}s")
    
    # === Directional Persistent Homology ===
    print("\nDirectional PH (8 directions):")
    start = time.time()
    feat = compute_ph_directional(dem_tile, n_samples=50, n_directions=8)
    timing['ph_directional'].append(time.time() - start)
    features['ph_directional'].append(feat)
    print(f"  → PH-Directional-800d: {timing['ph_directional'][-1]:.3f}s")

print(f"\n{'='*60}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*60}")

# Convert to arrays
feature_arrays = {key: np.array(vals) for key, vals in features.items()}
y = np.array(y_labels)

# Save everything with atomic writes
print(f"\nSaving feature matrices to {output_dir}...")
print("Using atomic writes to prevent corruption...")

# Map feature names to their arrays
save_mapping = {
    'testbed_features_traditional.npy': feature_arrays['traditional'],
    'testbed_features_ecc50.npy': feature_arrays['ecc50'],
    'testbed_features_ecc200.npy': feature_arrays['ecc200'],
    'testbed_features_betti100.npy': feature_arrays['betti100'],
    'testbed_features_betti200.npy': feature_arrays['betti200'],
    'testbed_features_landscape.npy': feature_arrays['landscape'],
    'testbed_features_persimage.npy': feature_arrays['persimage'],
    'testbed_features_ph_directional.npy': feature_arrays['ph_directional'],
    'testbed_labels.npy': y,
    'testbed_timing.npy': dict(timing),
}

for filename, data in save_mapping.items():
    filepath = output_dir / filename
    allow_pickle = filename == 'testbed_timing.npy'
    atomic_save(filepath, data, allow_pickle=allow_pickle)
    print(f"  ✓ {filename}")

print("\n✓ All features saved successfully!")
print(f"{'='*60}")
print("FEATURE DIMENSIONS:")
print(f"{'='*60}")

# Display feature dimensions in a clean format
dimension_display = {
    'Traditional': feature_arrays['traditional'],
    'ECC-50': feature_arrays['ecc50'],
    'ECC-200': feature_arrays['ecc200'],
    'Betti-100': feature_arrays['betti100'],
    'Betti-200': feature_arrays['betti200'],
    'Landscapes': feature_arrays['landscape'],
    'PersImages': feature_arrays['persimage'],
    'PH-Directional': feature_arrays['ph_directional'],
    'Labels': y,
}

for name, arr in dimension_display.items():
    print(f"{name:17s} {arr.shape}")

print(f"{'='*60}")
print(f"Output location: {output_dir}")

Computing features for 100 tiles...

Tile 1/100 (256_50_1)
✓ Traditional (150d): 0.141s
✓ ECC-50d: 0.054s
✓ ECC-200d: 0.053s

TDA Pipeline (Optimized):
  → PH Computation: 0.060s
  → Betti-200d vectorization: 0.004s
  → Betti-400d vectorization: 0.003s
  → Landscapes-1100d vectorization: 0.010s
  → PersImages-5000d vectorization: 0.003s
  → Total TDA: 0.080s

Directional PH (8 directions):
  → PH-Directional-800d: 0.239s

Tile 2/100 (256_50_2)
✓ Traditional (150d): 0.146s
✓ ECC-50d: 0.056s
✓ ECC-200d: 0.058s

TDA Pipeline (Optimized):
  → PH Computation: 0.064s
  → Betti-200d vectorization: 0.003s
  → Betti-400d vectorization: 0.000s
  → Landscapes-1100d vectorization: 0.010s
  → PersImages-5000d vectorization: 0.006s
  → Total TDA: 0.083s

Directional PH (8 directions):
  → PH-Directional-800d: 0.170s

Tile 3/100 (256_50_3)
✓ Traditional (150d): 0.123s
✓ ECC-50d: 0.062s
✓ ECC-200d: 0.055s

TDA Pipeline (Optimized):
  → PH Computation: 0.052s
  → Betti-200d vectorization: 0.005s
  → Be